# A2 · Leave-one-out for all k with a single distance matrix

**Type**: Implement from scratch · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Compute the leave-one-out accuracy of a KNN for **all the `k` from 1 to 60 in a single pass**: an n×n distance matrix with the diagonal excluded, a sort per row and the cumulated votes. Check
the exact equality with `cross_val_score(..., cv=LeaveOneOut())` and measure how much you save.

## Dataset

`load_breast_cancer` (569 nodules, 30 measurements, 2 classes) standardized once: `Z, y`. (Standardizing before a LOO is a small incorrectness that we accept here to isolate the problem: your goal is the algorithm.)

## Educational purpose

Exploit the structure of KNN: the predictions for `k` and for `k+1` share almost everything, so with a single sort of the distances you get **the whole validation curve** at the cost of a single evaluation. A standard trick
to choose `k` on small datasets.

## Task

1. `loo_matrix(Z)`: n×n matrix of Euclidean distances with the **diagonal set to `np.inf`** (a point cannot be its own neighbor).
2. `loo_accuracy(Z, y, k_max)`: array of `k_max` accuracies, for `k = 1..k_max`. Sort every row (`argsort`), take the labels of the `k_max` neighbors, build the cumulated counts per class with `np.cumsum` on a one-hot, and the prediction for `k` is the class with the most votes among the first `k` neighbors (on a tie, the smallest label: `argmax`).
3. Check: for `k` in `(1, 4, 7, 25)` the accuracy must match **exactly** `cross_val_score(KNeighborsClassifier(k), Z, y, cv=LeaveOneOut()).mean()`. Measure with `time.time()` `t_slow` (the four scikit-learn LOOs) and `t_fast` (your function with `k_max=60`) and save `speedup = t_slow / t_fast`.
4. Save `best_k` (the `k` with maximum accuracy; on a tie, the smallest) and `simple_rule_k`: the **maximum** `k` whose accuracy is within **one percentage point** of the maximum.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
Z = StandardScaler().fit_transform(X)
print(Z.shape, "| classes:", np.bincount(y))

## Your work

In [ ]:
def loo_matrix(Z):
    ...


def loo_accuracy(Z, y, k_max):
    ...

In [ ]:
t0 = time.time()
slow = {k: cross_val_score(KNeighborsClassifier(k), Z, y, cv=LeaveOneOut()).mean() for k in (1, 4, 7, 25)}
t_slow = time.time() - t0
t0 = time.time()
curve = loo_accuracy(Z, y, 60)
t_fast = time.time() - t0
speedup = t_slow / t_fast
print(f"scikit-learn (4 values of k): {t_slow:.1f}s | yours (60 values of k): {t_fast:.3f}s | ratio: {speedup:.0f}x")

In [ ]:
best_k = None
simple_rule_k = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`order = np.argsort(D, axis=1)[:, :k_max]`; `labels = y[order]` (n × k_max); `oh = np.eye(n_classes)[labels]` (n × k_max × n_classes); `votes = np.cumsum(oh, axis=1)`; the prediction with `k` neighbors is `votes[:, k - 1, :].argmax(1)`; `accuracy = (votes.argmax(2) == y[:, None]).mean(0)`.'):
    D = loo_matrix(Z)
    check(D.shape == (569, 569) and np.isinf(np.diag(D)).all() and np.isfinite(D[~np.eye(569, dtype=bool)]).all(), "569 x 569 matrix with the diagonal at infinity", "np.fill_diagonal(D, np.inf)")
    check(len(curve) == 60 and all(abs(curve[k - 1] - slow[k]) < 1e-12 for k in (1, 4, 7, 25)), "the LOO accuracy matches scikit-learn exactly for k = 1, 4, 7, 25", "votes among the first k neighbors, tie -> smallest label")
    check(speedup > 5, "your function is over 5 times faster (60 values of k against 4)", "a single sort of the rows")
    check(best_k == int(np.argmax(curve)) + 1, "best k: the one with maximum accuracy (on a tie, the smallest)", "np.argmax(curve) + 1")
    check(simple_rule_k == max(k for k in range(1, 61) if curve[k - 1] >= curve.max() - 0.01) and simple_rule_k >= best_k, "k of the simple rule: the largest within one point of the maximum", "smoother model at equal performance")
done()

---
The commented solution is in `solutions/A_from_scratch/A2_leave_one_out_for_all_k.ipynb`. Open it after you pass the check, or after you have really tried.